<img src="./logo_UTN.svg" align="right" width="150" /> 

### Teoría de Circuitos II

# Trabajo Semanal N°10

Autor: *Olea Gasalla Ignacio Ezequiel*

*Fecha de entrega: viernes 30 de octubre de 2026*

#### **Consigna** 
<img src="./consigna_1.png" /> 

##### **Anexo**:
📎 En caso de no poder acceder al documento, presione aquí: [Mostrar PDF](./tc2_ts10.pdf)

In [85]:
from IPython.display import IFrame
IFrame("./tc2_ts10.pdf", width=1000, height=1000)

In [86]:
# Importo las librerias
import numpy as np
import sympy as sp
from sympy import Eq
import scipy.signal as sig
from pytc2.sistemas_lineales import pretty_print_lti, tfcascade, pzmap, GroupDelay, bodePlot, pretty_print_SOS, tf2sos_analog, pretty_print_bicuad_omegayq,tf2zpk, parametrize_sos, analyze_sys
from pytc2.sintesis_dipolo import foster , cauer_LC
from pytc2.dibujar import dibujar_foster_serie , dibujar_foster_derivacion , dibujar_cauer_LC
from pytc2.general import s, w, print_latex, a_equal_b_latex_s, print_subtitle, Chebyshev_polynomials

In [87]:
# Elijo el orden y ripple sugerido:
order  = 3
ripple = 1

# Calculo el epsilon apartir del ripple: 
eps = np.sqrt( 10**(ripple/10) - 1 )

# Calculo |T|^2 siendo que S21 = T(s):
T = sp.expand(1 + eps**2 * (sp.expand(Chebyshev_polynomials(order)))**2)
w_ = list(T.free_symbols)[0] # Permite que .subs funcione 
T = T.subs(w_, s/sp.I)

# Dejo mónico el polinomio 
T_poly = sp.Poly(T, s)
cp = T_poly.LC()
T = (1/cp) / (T/cp)
display(Eq(sp.Function("|S_{21}|^2")(s), T))

# Aplico Darlinton para calular |S11|^2:
E = sp.cancel(1 - T)
num, den = sp.fraction(E)
display(Eq(sp.Function("|S_{11}|^2")(s), num/den))



#----------------------------------------------------------------------------------------------------------------------

# Asigno los valores de los coeficientes de |S11|^2:
num = np.array([ 1 , 0 , 1.5 , 0 , 0.5625 , 0 ,  0 ])
den = np.array([ 1 , 0 , 1.5 , 0 , 0.5625 , 0 , -0.241382255866351])

# Separo en dos los polos y ceros asumiendo |S11(s)|^2 = S11(s)*S11(-s)
cheby_zero = np.roots(num)
cheby_zero = cheby_zero[ np.real(cheby_zero) < 0 ]
cheby_zero = np.append(cheby_zero, 0) # Agrego uno de los Ceros dobles del origen 

cheby_roots = np.roots(den)
cheby_roots = cheby_roots[ np.real(cheby_roots) < 0 ]

# Consigo la Función Transferencia:
z = cheby_zero
p = cheby_roots
k = 1
num, den = sig.zpk2tf(z,p,k)
    # Elimino los valores de los coeficientes despreciables:
tol = 1e-9
num[np.abs(num) < tol] = 0
den[np.abs(den) < tol] = 0
mi_tf = sig.TransferFunction(num,den)

print('---------------------------------------------------')
print('S11=')
#analyze_sys( mi_tf )
pretty_print_lti(num,den)
this_sos = tf2sos_analog(num, den)
pretty_print_SOS(this_sos, mode='omegayq')


print('---------------------------------------------------')
numZ1 = np.trim_zeros(num + den, 'f')
denZ1 = np.trim_zeros(-num + den, 'f')

NumZ1 = (sum(c*s**(len(numZ1)-1-i) for i,c in enumerate(numZ1))) /( 2)
DenZ1 = sum(c*s**(len(denZ1)-1-i) for i,c in enumerate(denZ1))
Z1 = sp.cancel(NumZ1/DenZ1)

Z1_display = Z1.xreplace({
    atom: sp.Float(round(float(atom), 2), 3)
    for atom in Z1.atoms(sp.Float)
})
display(Eq(sp.Function("Z_1")(s), (2)*Z1_display))


Eq(|S_{21}|^2(s), -0.241382255866351/(1.0*s**6 + 1.5*s**4 + 0.5625*s**2 - 0.241382255866351))

Eq(|S_{11}|^2(s), (1.0*s**6 + 1.5*s**4 + 0.5625*s**2)/(1.0*s**6 + 1.5*s**4 + 0.5625*s**2 - 0.241382255866351))

---------------------------------------------------
S11=


<IPython.core.display.Math object>

<IPython.core.display.Math object>

---------------------------------------------------


Eq(Z_1(s), 2*(1.0*s**3 + 0.49*s**2 + 0.99*s + 0.25)/(0.99*s**2 + 0.49*s + 0.49))

### Simulación en LTSpice

 Se utilizó el siguiente circuito (obtenido al usar Cauer como se vió en el documento superior) para graficar los parámetros S11 y S21:
<img src="./circuit.png" /> 


 En la siguiente imagen, se muestra el gráfico de ambos parámetros:
<img src="./study.png" /> 

De esta forma se puede explicar el comportamiento del circuito:
 Durante la *Banda de paso* se ve que el parámetro S11 va en aumento siendo que en casi continua solo se refleja un 35% de la señal mientras que, cuando se llega a la *frecuencia de corte* se tiene una reflexión del 50%, ocurre un pico inferior interesante que se puede asumir que es cuando las bobinas y el capacitor entra en resonancia, anulando sus aportes de impedancias y dejando más adaptado el circuito. Luego ocurre la transición que la reflexión pasa de un 30% como su mínimo a más del 90% en un par de mHz y, en la banda de parada, se tiene una reflexión del 100%.


Aplicando las reglas para desnormalizar los componentes por un nivel de impedancia y una frecuencia se consiguió el siguiente circuito:

<img src="./circuit_2.png" /> 

#### Parte B: Trabajo de Investigación sobre NANO VNA ( Vector Network Analyzer )
1. Explique brevemente la función principal de un VNA y en que situación/es puede ser fundamental tenerlo como herramienta de medición. 
2. Comparar respecto a un DSO GW Insteak características de: Ancho de Banda, Rango Dinámico e Impedancia de Entrada.
3. Además de las mediciones de los parametros  , investigar si el equipo cuenta con alguna medición en el dominio del tiempo.

1) La función principal del VNA es la analizar como en una red modifica la amplitud y la fase de las señales que inciden o se reflajan en ella utilizando los parámetros de dispersión (el NANO-VNA mide los parámetros S11 y S21 en particular). Con esa información, se puede obtener los otros parámetros y se puede caracterizar una red por su impedancia o admitancia (incluso siendo complejas), por las la pérdida de inserción o de retorno o por su relación de onda estacionaria (su coeficiente de reflexión o su ROE).
   
   Es fundamental tenerlo como herramienta cuando se trabajan a altas frecuencias, donde los efectos de reflexión y adaptación de impedancias adquieren una importancia considerable ya sea en amplificadores o filtros, cables y conectores o incluso antenas.

2) El NANO VNA hace un barrido en frecuencia desde los 50kHz a los 4.4GHz, mientras que los modelos de DSO planteados pueden llegar a tener un ancho de banda de hasta unos 150MHz en el mejor de los casos.

   Si hablamos del rango dinámico como la mínima amplitud que se puede visualizar, el DSO tiene 2mV/div con un display de 8 divisiones verticales. Mientras que el VNA tiene un rango dinámico de 80 dB a 3 GHz (bajar la frecuencia mejora el rango dinámico).

   La impedancia con la que cuenta el DSO es muy alta para intentar no cargar el circuito al conectarlo en paralelo, en el caso del GW Insteak, tiene un impedancia de entrada de "1MΩ±2%, ~15pF" implicando que tiene impedancia de entrada compleja. Por otro lado, la impedancia de entrada del VNA es de unos 50Ω nominales ya que trabaja con puertos de radiofrecuencia.

3. El NANO VNA cuenta con una función llamada "Time Domain Reflectometry" (o "Reflectometría en el Dominio del Tiempo" en español) que permite realizar la antitransformada de Fourier para visualizar la señal en el tiempo. Esta función tambien incluye sub-funciones (según el modelo) para conseguir la respuesta al impulso o la respuesta al escalon del sistema. Aun así, existe un compromiso entre la resolución y la distancia máxima que puede medirse debido a las limitaciones de instrumento.